In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os
import pprint

In [2]:
load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-5"

In [6]:
tools = [
    {
        "name": "extract_return_request",
        "description": "Extract a structured return request from a customer support message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The customer's order ID, or null if not provided.",
                },
                "item": {
                    "type": ["string", "null"],
                    "description": "The item the customer wants to return or replace.",
                },
                "reason": {
                    "type": "string",
                    "enum": ["normal_return", "damaged_item", "billing_dispute",
                             "policy_exception", "unclear", "other"],
                    "description": "The main reason for the request.",
                },
                "reason_detail": {
                    "type": ["string", "null"],
                    "description": "Extra detail when the reason is other or unclear.",
                },
                "desired_action": {
                    "type": "string",
                    "enum": ["refund", "replacement", "exchange",
                             "store_credit", "unclear", "other"],
                    "description": "What the customer wants as a resolution.",
                },
                "desired_action_detail": {
                    "type": ["string", "null"],
                    "description": "Extra detail when desired_action is unclear or other.",
                },
                "evidence_provided": {
                    "type": "boolean",
                    "description": "Whether the customer already provided evidence, "
                                   "such as photos or proof of purchase.",
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "normal", "high", "unclear"],
                    "description": "How time-sensitive the request is, based on what "
                                   "the customer states or implies.",
                },
                "missing_information": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Information needed before the request can be processed.",
                },
            },
            "required": [
                "order_id", "item", "reason", "reason_detail",
                "desired_action", "desired_action_detail",
                "evidence_provided", "urgency", "missing_information",
            ],
        },
    }
]

In [10]:
customer_message = """
I bought this 6 months ago and I never used it.  I want to return it.
"""

message = client.messages.create(
    model=model,
    max_tokens=800,
    output_config={"effort": "low"},
    tools=tools,
    tool_choice={
        "type": "tool",
        "name": "extract_return_request"
    },
    messages=[
        {
            "role": "user",
            "content": customer_message

        }
    ]
)

print(message.content[0].model_dump_json(indent=2))

{
  "id": "toolu_01CHzXFmN79uwk4kQf91Jqnm",
  "caller": {
    "type": "direct"
  },
  "input": {
    "order_id": null,
    "item": null,
    "reason": "normal_return",
    "reason_detail": "Customer purchased item 6 months ago and never used it, now wants to return it.",
    "desired_action": "refund",
    "desired_action_detail": null,
    "evidence_provided": false,
    "urgency": "low",
    "missing_information": [
      "order_id",
      "item description",
      "proof of purchase",
      "confirmation of desired resolution (refund vs store credit vs exchange)",
      "eligibility given 6-month timeframe (may exceed standard return window)"
    ]
  },
  "name": "extract_return_request",
  "type": "tool_use"
}


In [ ]:
def validate_return_request(extraction):
    errors = []
    allowed = {"refund", "replacement", "store_credit", "unclear"}

    # fixable and can be retried
    if extraction["desired_action"] not in allowed:
        errors.append("desired_action invalid")

    # absent - flag, do not invent
    if extraction["order_id"] is None:
        extraction["missing_information"].append("order_id")

    # risky - flag, do not invent
    if extraction["conflict_detected"] or extraction["desired_action_confidence"] == "low":
        extraction["human_review_requred"] = True

    return errors, extraction

In [5]:
def needs_human_review(data):
    if data["confidence"] < 0.7:
        return True

    if data["reason"] in ["unclear", "policy_exception"]:
        return True

    if data["human_review_required"]:
        return True

    return False